## Limitations
- Dataset is sampled (20,000 out of the full dataset (1,000,000 data)); results may vary with different random states.
- Burnout is an inherently subjective measure; not all contributing factors 
  (e.g. personality, coping mechanisms) are captured in the available features.
- Model A has R² score of around 0.72, indicating that approximately 28% of burnout variance 
  remains unexplained by the selected features.
- Predictions should be interpreted as indicative estimates, not clinical assessments.

In [11]:
import pandas as pd

In [12]:
df = pd.read_csv("datasets/academic_stress_level.csv")
df = df.sample(n=20000, random_state=42)

In [13]:
# Keeping the used columns
cols_to_keep = [
    "study_hours_per_day",
    "sleep_hours",
    "exam_pressure",
    "stress_level",
    "financial_stress",
    "social_support",
    "anxiety_score",
    "depression_score",
    "family_expectation",
    "burnout_score",
]

df = df[cols_to_keep]

In [14]:
# Data cleaning
num_cols = df.select_dtypes("number").columns

for col in num_cols:
    q3 = df[col].quantile(0.75)
    q1 = df[col].quantile(0.25)
    iqr = q3 - q1
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]

print(f'After cleaning: {len(df)}')

After cleaning: 19567


In [15]:
x = df.drop(columns=["burnout_score"]) # Dropping burnout_score as it's the target feature
y = df["burnout_score"]

In [16]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [17]:
from sklearn.ensemble import GradientBoostingRegressor

model_a = GradientBoostingRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.7,  
    max_features='sqrt',  
    n_iter_no_change=50,  
    validation_fraction=0.1,  
    tol=1e-4,
    random_state=42,
    min_samples_leaf=5
)

model_a.fit(x_train, y_train)
y_pred = model_a.predict(x_test)

In [18]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE  : {mse}")
print(f"RMSE : {rmse}")
print(f"MAE  : {mae}")
print(f"R2   : {r2}")

joblib.dump(model_a, "models/modelA.pkl")
print("Model A is successfully saved!")

MSE  : 0.7129572461708263
RMSE : 0.8443679566224824
MAE  : 0.6431403429209612
R2   : 0.7207556526499823
Model A is successfully saved!


In [19]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2   = cross_val_score(model_a, x, y, cv=kf, scoring="r2")
cv_mae  = cross_val_score(model_a, x, y, cv=kf, scoring="neg_mean_absolute_error")
cv_rmse = cross_val_score(model_a, x, y, cv=kf, scoring="neg_root_mean_squared_error")

print(f"CV R²   : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"CV MAE  : {(-cv_mae).mean():.4f} ± {(-cv_mae).std():.4f}")
print(f"CV RMSE : {(-cv_rmse).mean():.4f} ± {(-cv_rmse).std():.4f}")

CV R²   : 0.7172 ± 0.0090
CV MAE  : 0.6481 ± 0.0081
CV RMSE : 0.8491 ± 0.0101
